# Finding ORCIDs for students in the Biology and Geology of Tropical Islands Class between 1992 and 2019
Given a list of student author names we can search for ORCID metadata to help match names to correct ORCIDs. The metadata we have from ORCID profiles includs: other names, employment, education, works, peer-reviews, funding and email. In order for these metadata to be helpful, the ORCID metadata data must 1) exist in the profile and 2) be open to the public. If these data are available, it is not unusual for there to be multiple matches to many names, after all, ORCIDs exist to help identify the right person with a particular name.

This notebook provides some ORCID metadata that might help find ORCIDs, i.e. unique and persistent identifiers, for students  

To run this notebook (from your drive):
1.   In Colab press Commands in the menu above, input "out" and select **Clear all outputs** to make sure there are no old outputs. In Jupyter just press **Clear all outputs** in the menu bar.
2.   Press the arrows in the cells below to run the code in the cells. All cells with arrows must be run in order to make the notebook work!
3.   The code in these cells is hidden to make the notebook easier to read (hopefully) and help you focus on results. If you are interested, press the "Show Code" links to see code.

In [1]:
#@title Step 1. Set up the environment for running the notebook
import pandas as pd
from   tabulate import tabulate                     # makes pretty tables in many formats from dataframes
from   IPython.display import display, Markdown     # allows computation results to be displayed in markdown
import  ipywidgets as widgets                        # interactive widgets
#
# data source can be a git repository or a local directory.
#
dataSource = 'https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/'  # git repository


In [ ]:
#@title Step 2. Read the data the notebook needs: the index of authors and papers and the ORCID metadata found automatically
#
# Read data from the source
#
dataDate = '20260312_16'
if dataSource.startswith('https'):
    #
    # Data and metadata related to these papers are in a public git repository defined by dataSource
    # The index of the papers (a spreadsheet) is in this git repository as a csv file
    # read the index of student papers from git repository
    #
    url = f'{dataSource}/studentFullNames.csv'
    index_df = pd.read_csv(url)
    if not 'Authors Primary' in index_df.columns:
        raise ValueError(f"Expected column 'Authors Primary' not found in index dataframe from {url}")
    #
    # Read all sheets from the ORCID metadata files in the git repository
    # These are csv files spilt out from the ORCID metadata spreadsheet (xlsx) that is in the git repository
    #
    orcid_metadata_l = ['counts', 'employment', 'education', 'works', 'funding', 'email']
    orcid_dfs = {}
    for sheet in orcid_metadata_l:
        url = f"{dataSource}/orcidMetadata_{sheet}__{dataDate}.csv"
        orcid_dfs[sheet] = pd.read_csv(url)
else:
    #
    # data for this analysis are in local files (not for use in collab)
    # Read the index of student papers from excel, which includes author names and publication years, into a dataframe
    index_file = '/Users/metadatagamechanger/ProjectsAndPlans/FieldStations/MooreaStudentPapers/index_to_moorea_class_projects_1992-2022.xlsx'
    index_df = pd.read_excel(index_file, sheet_name='MooreaClass-Export.txt', engine='openpyxl')
    #
    # Read all sheets from the ORCID metadata file (xlsx) into a dictionary of dataframes
    #
    orcid_metadata_file = f'/Users/metadatagamechanger/Repositories/MooreaStudentPapers/data/orcidMetadata__{dataDate}.xlsx'
    orcid_dfs = pd.read_excel(orcid_metadata_file, sheet_name=None)
#
# Display the input data to verify they were read correctly
print(f"{len(index_df)} rows in the index dataframe")           # Display the index count
print(f"Sheets found: {list(orcid_dfs.keys())} ")

460 rows in the index dataframe
Sheets found: ['counts', 'employment', 'education', 'works', 'funding', 'email'] 


## Match Counts
An ORCID search for a name can return any number of matches, i.e. one for each person with that name that has an ORCID. The difficulty of picking the right ORCID increases as the number of matches increases. The counts dataframe in the ORCID output has the input names in a column named 'input'. The number of times these names occur is the number of matches for the name. Add these counts to the counts dataframe as the matchCounts column.

In [3]:
#@title Step 3. Compare the ORCID metadata with the index of papers to count the number of matches for each input name, and add it to the data
#
# Create a series of counts of the number of matches for each input name, and add it to the dataframe
#
counts_df                = orcid_dfs['counts']                                                                                 # get the counts dataframe from the dictionary of dataframes
matchCounts              = counts_df[counts_df['orcid'] != 'Not Found']['input'].value_counts()      # a series with the number of matches for each input name
counts_df['matchCounts'] = counts_df['input'].map(matchCounts).fillna(0).astype(int)                 # add the match counts to the dataframe, filling in 0 for names with no matches
#counts_df.fillna('', inplace=True)                                                                  # replace NaN with empty string for better display
#counts_df.head(20)                                                                                  # display the first 20 rows of the dataframe with match counts
print(f"{len(counts_df)} rows in the counts dataframe with match counts added")

3720 rows in the counts dataframe with match counts added


## Define your study year.
Each year of the class has roughly 20 students which seems like a managable number for the re-curation process. 
yzour year is defined here:

In [4]:
#@title Step 4. Define the year that you are re-curating
studyYear = 1992                        # pick a year to focus on


In [8]:
#@title Step 5. Select the data for your year, and display the authors with match counts less than or equal to a limit
matchCountLimit = 10                        # define the maximum number of matches to be explored (default = 10)
display(Markdown(f'##Study year{studyYear} with matches <= {matchCountLimit}**'))
studyYearIndex_df = index_df[index_df['Pub Year'] == studyYear]                    # the publication years are in the index dataframe and not in counts_df
                                                                                   # studyYearIndex is the index for papers published during the study year.
studyYearAuthor_l       = list(studyYearIndex_df['Authors, Primary'].values)            # studyYearAuthor_l is the list of authors for the selected year
authorORCIDMetadata_df  = counts_df[counts_df['input'].isin(studyYearAuthor_l)]         # get the counts dataframe for the authors in the selected year

matchCountLimit = 10                                                                        # define the maximum number of counts to be explored (default = 10)
df = authorORCIDMetadata_df[authorORCIDMetadata_df['matchCounts'] <= matchCountLimit]       # select the rows with match counts less than or equal to the limit
#df.columns = ['input', 'given', 'family', 'orcid', 'matchCounts', 'other-names', 'employment',
#            'education', 'works', 'peer-reviews', 'funding', 'emails',
#            'other_names', 'Total', 'journal']
#df.matchCounts = df.matchCounts.astype(int)
display(Markdown(f'**{len(df)} authors in {studyYear} with matches <= {matchCountLimit}**'))
display(Markdown(tabulate(df.sort_values(by='matchCounts'), headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))


##Study year1992 with matches <= 10**

**25 authors in 1992 with matches <= 10**

| input                | given       | family    | orcid               |   other-names |   employment |   education |   works |   peer-reviews |   funding |   emails |   other_names |   Total | journal                                                     |   matchCounts |
|:---------------------|:------------|:----------|:--------------------|--------------:|-------------:|------------:|--------:|---------------:|----------:|---------:|--------------:|--------:|:------------------------------------------------------------|--------------:|
| Gregory Battersby    | Gregory     | Battersby | Not Found           |           nan |          nan |         nan |     nan |            nan |       nan |      nan |           nan |       0 | nan                                                         |             0 |
| Sam Shiley           | Sam         | Shiley    | Not Found           |           nan |          nan |         nan |     nan |            nan |       nan |      nan |           nan |       0 | nan                                                         |             0 |
| Sharon London        | Sharon      | London    | Not Found           |           nan |          nan |         nan |     nan |            nan |       nan |      nan |           nan |       0 | nan                                                         |             0 |
| Lorien A. Ferris     | Lorien A.   | Ferris    | Not Found           |           nan |          nan |         nan |     nan |            nan |       nan |      nan |           nan |       0 | nan                                                         |             0 |
| Gretchen Elmquist    | Gretchen    | Elmquist  | Not Found           |           nan |          nan |         nan |     nan |            nan |       nan |      nan |           nan |       0 | nan                                                         |             0 |
| Raj Denhoy           | Raj         | Denhoy    | Not Found           |           nan |          nan |         nan |     nan |            nan |       nan |      nan |           nan |       0 | nan                                                         |             0 |
| Tegan Paige Churcher | Tegan Paige | Churcher  | Not Found           |           nan |          nan |         nan |     nan |            nan |       nan |      nan |           nan |       0 | nan                                                         |             0 |
| Laurie Tucker        | Laurie      | Tucker    | Not Found           |           nan |          nan |         nan |     nan |            nan |       nan |      nan |           nan |       0 | nan                                                         |             0 |
| Leslie Engel         | Leslie      | Engel     | 0000-0002-9774-238X |             0 |            0 |           0 |       0 |              0 |         0 |      nan |           nan |       0 | nan                                                         |             1 |
| Jeffrey S. Shima     | Jeffrey     | Shima     | 0000-0001-5770-4859 |             3 |            1 |           2 |      58 |              4 |         3 |      nan |           nan |      66 | Ecology                                                     |             1 |
| Jennifer R. Graff    | Jennifer    | Graff     | 0000-0002-2931-9547 |             0 |            1 |           0 |       0 |              0 |         0 |      nan |           nan |       1 | nan                                                         |             2 |
| Louis Santiago       | Louis       | Santiago  | 0000-0001-5994-6122 |             0 |            1 |           0 |     127 |            172 |         0 |      nan |           nan |     300 | Tree Physiology                                             |             2 |
| Louis Santiago       | Louis       | Santiago  | 0000-0002-6943-7177 |             0 |            0 |           0 |       0 |              0 |         0 |      nan |           nan |       0 | nan                                                         |             2 |
| Marc Kramer          | Marc M.     | Kramer    | 0000-0003-2636-8658 |             0 |            1 |           0 |      10 |              0 |         0 |      nan |           nan |      11 | Journal of Pension Economics and Finance                    |             2 |
| Marc Kramer          | marc        | kramer    | 0000-0002-8022-4527 |             0 |            0 |           0 |       0 |              0 |         0 |      nan |           nan |       0 | nan                                                         |             2 |
| Jennifer R. Graff    | Jennifer    | Graff     | 0000-0001-8261-3158 |             0 |            0 |           0 |       0 |              0 |         0 |      nan |           nan |       0 | nan                                                         |             2 |
| Michael A. Gilchrist | Michael     | Gilchrist | 0000-0002-6936-0771 |             0 |            1 |           0 |      34 |              4 |         0 |      nan |           nan |      39 | Journal of Theoretical Biology                              |             3 |
| Michael A. Gilchrist | Michael     | Gilchrist | 0000-0001-9762-1951 |             0 |            0 |           0 |       0 |              0 |         0 |      nan |           nan |       0 | nan                                                         |             3 |
| Michael A. Gilchrist | Michael     | Gilchrist | 0000-0003-1765-429X |             1 |            4 |           4 |     171 |              0 |         0 |      nan |           nan |     175 | Computer Methods in Biomechanics and Biomedical Engineering |             3 |
| Jeffrey Becker       | Jeffrey     | Becker    | 0000-0002-0061-1392 |             0 |            1 |           0 |       0 |              0 |         0 |      nan |           nan |       1 | nan                                                         |             6 |
| Jeffrey Becker       | Jeffrey     | Becker    | 0000-0001-7645-2466 |             0 |            0 |           0 |      27 |              0 |         0 |      nan |           nan |      27 | Molecular Cancer Therapeutics                               |             6 |
| Jeffrey Becker       | Jeffrey     | Becker    | 0000-0003-0467-5913 |             0 |            0 |           0 |       2 |              1 |         0 |      nan |           nan |       3 | G3&amp;#58; Genes|Genomes|Genetics                          |             6 |
| Jeffrey Becker       | Jeffrey     | Becker    | 0000-0002-9799-7243 |             0 |            0 |           0 |       0 |              0 |         0 |      nan |           nan |       0 | nan                                                         |             6 |
| Jeffrey Becker       | Jeffrey     | Becker    | 0009-0007-6442-9462 |             0 |            1 |           0 |       2 |              3 |         0 |      nan |           nan |       6 | Intermetallics                                              |             6 |
| Jeffrey Becker       | Jeffrey     | Becker    | 0000-0001-8759-3774 |             0 |           12 |           3 |      22 |              0 |         0 |        1 |           nan |      35 | Smarthistory.org                                            |             6 |

In [9]:
#@title Step 6. Select an author to look at in more detail
author_names = df['input'].unique().tolist()                    # create a list of unique author names with >= matchCounts from the selected year
author_widget = widgets.Dropdown(                               # make a pull-down widget with the author names
    options=author_names,
    description='Select Author:',
    style={'description_width': '100px'}
)
display(author_widget)                                          # display the widget to make selection

def on_author_change(change):
    global selectedAuthor
    selectedAuthor = change['new']

author_widget.observe(on_author_change, names='value')          # set up the widget to call the on_author_change function when a new author is selected   
selectedAuthor = author_widget.value                            # get the value of the selected author from the widget   


Dropdown(description='Select Author:', options=('Gregory Battersby', 'Jeffrey Becker', 'Tegan Paige Churcher',…

In [11]:
#@title Step 7. Display ORCID Metadata for that author
element_l = ['email','education', 'employment', 'works','funding']

input = selectedAuthor
orcid_l = list(authorORCIDMetadata_df[authorORCIDMetadata_df['input']==input]['orcid'].values)

if orcid_l[0] == 'Not Found':
    print(f'No ORCID was found for {input} from the {studyYear} class')
else:
    if len(orcid_l) == 1:
        display(Markdown(f'{input} from the {studyYear} class has 1 ORCID match: [{orcid_l[0]}](https://commons.datacite.org/orcid.org/{orcid_l[0]})'))
    else:
        print(f'{input} from the {studyYear} class has {len(orcid_l)} ORCID matchs: ', ', '.join(orcid_l))
    for element in element_l:
        data = orcid_dfs[element]
        df = data[data['orcid'].isin(orcid_l)]
        if len(df) > 0:
            df = df.fillna('')
            display(Markdown(f'**{input} from the {studyYear} class {element}: ({len(df)})**'))
            display(Markdown(tabulate(df, headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))
        else:
            display(Markdown(f'{input} from the {studyYear} class has no {element} metadata.'))

Louis Santiago from the 1992 class has 2 ORCID matchs:  0000-0002-6943-7177, 0000-0001-5994-6122


Louis Santiago from the 1992 class has no email metadata.

**Louis Santiago from the 1992 class education: (2)**

| input          | given   | family   | orcid               | name   | identifier   | department   | degree   | start   | end   |
|:---------------|:--------|:---------|:--------------------|:-------|:-------------|:-------------|:---------|:--------|:------|
| Louis Santiago | Louis   | Santiago | 0000-0002-6943-7177 |        |              |              |          |         |       |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 |        |              |              |          |         |       |

**Louis Santiago from the 1992 class employment: (2)**

| input          | given   | family   | orcid               | name                                | identifier                           | start   | end   |
|:---------------|:--------|:---------|:--------------------|:------------------------------------|:-------------------------------------|:--------|:------|
| Louis Santiago | Louis   | Santiago | 0000-0002-6943-7177 |                                     |                                      |         |       |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | University of California, Riverside | http://dx.doi.org/10.13039/100007602 | 2006.0  |       |

**Louis Santiago from the 1992 class works: (128)**

| input          | given   | family   | orcid               | source                                    | title                                                                                                                                                                                                                                                    | journal                                                                         | publicationDate   | identifier                         |
|:---------------|:--------|:---------|:--------------------|:------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------|:------------------|:-----------------------------------|
| Louis Santiago | Louis   | Santiago | 0000-0002-6943-7177 |                                           |                                                                                                                                                                                                                                                          |                                                                                 |                   |                                    |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago                            | Leaf drought and heat tolerance are integrated across three temperate biome types                                                                                                                                                                        | Scientific Reports                                                              | 2025.0            | 10.1038/s41598-025-95623-5         |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago                            | Leaf Turgor Loss Does Not Coincide With Cell Plasmolysis in Drought‐Tolerant Chaparral Species                                                                                                                                                           | Plant, Cell &amp; Environment                                                   | 2025.0            | 10.1111/pce.15505                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago                            | Physiological Symptoms Induced by Drought Stress Outweigh Vascular Pathogen Infection in Walnut                                                                                                                                                          | Tree Physiology                                                                 | 2025.0            | 10.1093/treephys/tpaf034           |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref                                  | Drought stress influences foraging preference of a solitary bee on two wildflowers                                                                                                                                                                       | Annals of Botany                                                                | 2025.0            | 10.1093/aob/mcae048                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Supporting inclusive scientific communities: Insights from the ATBC society survey                                                                                                                                                                       | Biotropica                                                                      | 2025.0            | 2-s2.0-85205935119                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Daily and Seasonal Changes of Sap Flow in Gamhong Apple Cultivar and Estimate the Tree-Level Transpiration Using Penman-Monteith Reference Evapotranspiration                                                                                            | SSRN                                                                            | 2024.0            | 2-s2.0-85208997918                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Daily and seasonal changes of sap flow in Gamhong apple cultivar and estimate the tree-level transpiration using Penman-Monteith reference evapotranspiration                                                                                            | South African Journal of Botany                                                 | 2024.0            | 10.1016/j.sajb.2024.08.042         |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Initial stomatal conductance increases photosynthetic induction of trees leaves more from sunlit than from shaded environments: a meta-analysis                                                                                                          | Tree Physiology                                                                 | 2024.0            | 2-s2.0-85208772707                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Revealing genetic determinants of photosynthesis-related traits in citrus via genome-wide association studies                                                                                                                                            | Fruit Research                                                                  | 2024.0            | 2-s2.0-85195146959                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | The benefits of woody plant stem photosynthesis extend to hydraulic function and drought survival in Parkinsonia florida                                                                                                                                 | Tree Physiology                                                                 | 2024.0            | 2-s2.0-85187124374                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago,Scopus - Elsevier          | Effective rhizobia enhance legume growth during subsequent drought despite water costs associated with nitrogen fixation                                                                                                                                 | Plant and Soil                                                                  | 2023.0            | 2-s2.0-85165990667                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago,Scopus - Elsevier          | Exploring the Phylogenetic Relationship among Citrus through Leaf Shape Traits: A Morphological Study on Citrus Leaves                                                                                                                                   | Horticulturae                                                                   | 2023.0            | 2-s2.0-85166257458                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago,Scopus - Elsevier          | Biocrust carbon exchange varies with crust type and time on Chihuahuan Desert gypsum soils                                                                                                                                                               | Frontiers in Microbiology                                                       | 2023.0            | 2-s2.0-85159961219                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Irrigated urban trees exhibit greater functional trait plasticity compared to natural stands                                                                                                                                                             | Biology Letters                                                                 | 2023.0            | 2-s2.0-85145428829                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Plant physiological indicators for optimizing conservation outcomes                                                                                                                                                                                      | Conservation Physiology                                                         | 2023.0            | 10.1093/conphys/coad073            |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago,Scopus - Elsevier          | An expanded framework for wildland–urban interfaces and their management                                                                                                                                                                                 | Frontiers in Ecology and the Environment                                        | 2022.0            | 2-s2.0-85132153574                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago,Scopus - Elsevier          | Convergence in phosphorus constraints to photosynthesis in forests around the world                                                                                                                                                                      | Nature Communications                                                           | 2022.0            | 10.1038/s41467-022-32545-0         |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Large variation in availability of Maya food plant sources during ancient droughts                                                                                                                                                                       | Proceedings of the National Academy of Sciences of the United States of America | 2022.0            | 10.1073/pnas.2115657118            |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Time will tell: towards high-resolution temporal tree-ring isotope analyses                                                                                                                                                                              | Tree Physiology                                                                 | 2022.0            | 2-s2.0-85153111097                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago,Crossref,Scopus - Elsevier | Effects of temperature, soil moisture and light intensity on the temporal pattern of floral gene expression and flowering of avocado buds (Persea americana cv. Hass)                                                                                    | Scientia Horticulturae                                                          | 2021.0            | 10.1016/j.scienta.2021.109940      |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Louis Santiago,Scopus - Elsevier          | Species-specific performance and trade-off between growth and survival in the early-successional light-demanding group                                                                                                                                   | Photosynthetica                                                                 | 2021.0            | 2-s2.0-85104827123                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Hydraulic traits of Neotropical canopy liana and tree species across a broad range of wood density: Implications for predicting drought mortality with models                                                                                            | Tree Physiology                                                                 | 2021.0            | 10.1093/treephys/tpaa106           |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Shade tree species affect gas exchange and hydraulic conductivity of cacao cultivars in an agroforestry system                                                                                                                                           | Tree Physiology                                                                 | 2021.0            | 10.1093/treephys/tpaa119           |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Stem functional traits, not just morphology, explain differentiation along the liana-tree continuum                                                                                                                                                      | Tree Physiology                                                                 | 2021.0            | 10.1093/treephys/tpab117           |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Towards a statistically robust determination of minimum water potential and hydraulic risk in plants                                                                                                                                                     | New Phytologist                                                                 | 2021.0            | 2-s2.0-85110780064                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Unraveling the relative role of light and water competition between lianas and trees in tropical forests: A vegetation model analysis                                                                                                                    | Journal of Ecology                                                              | 2021.0            | 2-s2.0-85096870017                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Contrasting adaptation of xylem to dehydration in two Vitis vinifera L. sub-species                                                                                                                                                                      | Vitis - Journal of Grapevine Research                                           | 2020.0            | 10.5073/vitis.2020.59.53-61        |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Functional traits of leaves and photosynthetic stems of species from a sarcocaulescent scrub in the southern Baja California Peninsula                                                                                                                   | American Journal of Botany                                                      | 2020.0            | 10.1002/ajb2.1546                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Going underground: new approaches to assess dynamic root behaviour during drought                                                                                                                                                                        | New Phytologist                                                                 | 2020.0            | 2-s2.0-85074967325                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Small biodiversity effects on leaf litter production of a seasonal heath vegetation                                                                                                                                                                      | Journal of Vegetation Science                                                   | 2020.0            | 2-s2.0-85087167237                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Physiological Responses of Onion Varieties to varying Photoperiod and Temperature Regimes                                                                                                                                                                | Agriculture                                                                     | 2019.0            | 10.3390/agriculture9100214         |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Atlantic forest and leaf traits: an overview                                                                                                                                                                                                             | Trees - Structure and Function                                                  | 2019.0            | 10.1007/s00468-019-01864-z         |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Bayesian inference of hydraulic properties in and around a white fir using a process-based ecohydrologic model                                                                                                                                           | Environmental Modelling and Software                                            | 2019.0            | 2-s2.0-85061338538                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Costs and benefits of photosynthetic stems in desert species from southern California                                                                                                                                                                    | Functional Plant Biology                                                        | 2019.0            | 10.1071/FP18203                    |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Large hydraulic safety margins protect Neotropical canopy rainforest tree species against hydraulic failure during drought                                                                                                                               | Annals of Forest Science                                                        | 2019.0            | 10.1007/s13595-019-0905-0          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Modeling of xylem vessel occlusion in grapevine                                                                                                                                                                                                          | Tree Physiology                                                                 | 2019.0            | 2-s2.0-85071704997                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | The physiological response of ‘Hass’ avocado to salinity as influenced by rootstock                                                                                                                                                                      | Scientia Horticulturae                                                          | 2019.0            | 10.1016/j.scienta.2019.108629      |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Traits uncover quasi-neutral community assembly in a coastal heath vegetation                                                                                                                                                                            | Journal of Plant Ecology                                                        | 2019.0            | 2-s2.0-85084550943                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Climate and soils together regulate photosynthetic carbon isotope discrimination within C<sub>3</sub> plants worldwide                                                                                                                                   | Global Ecology and Biogeography                                                 | 2018.0            | 10.1111/geb.12764                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Coordination and trade-offs among hydraulic safety, efficiency and drought avoidance traits in Amazonian rainforest canopy tree species                                                                                                                  | New Phytologist                                                                 | 2018.0            | 10.1111/nph.15058                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Diurnal patterns of photosynthesis and water relations for four orchard-grown pomegranate (Prunica granatum L.) cultivars                                                                                                                                | Journal of the American Pomological Society                                     | 2018.0            | 2-s2.0-85052875656                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Evaluation of leaf carbon isotopes and functional traits in avocado reveals water-use efficient cultivars                                                                                                                                                | Agriculture, Ecosystems and Environment                                         | 2018.0            | 10.1016/j.agee.2018.04.021         |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Isotopic composition of leaf carbon (δ<sup>13</sup>C) and nitrogen (δ<sup>15</sup>N) of deciduous and evergreen understorey trees in two tropical Brazilian Atlantic forests                                                                             | Journal of Tropical Ecology                                                     | 2018.0            | 2-s2.0-85045619940                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Orchard establishment, precocity, and eco-physiological traits of several pomegranate cultivars                                                                                                                                                          | Scientia Horticulturae                                                          | 2018.0            | 2-s2.0-85043374406                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Stomatal behaviour and stem xylem traits are coordinated for woody plant species under exceptional drought conditions                                                                                                                                    | Plant Cell and Environment                                                      | 2018.0            | 10.1111/pce.13367                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | X international workshop on sap flow                                                                                                                                                                                                                     | Acta Horticulturae                                                              | 2018.0            | 2-s2.0-85059583305                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Crossref,Scopus - Elsevier                | Functional strategies of tropical dry forest plants in relation to growth form and isotopic composition                                                                                                                                                  | Environmental Research Letters                                                  | 2017.0            | 2-s2.0-85036467863                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | High N, dry: Experimental nitrogen deposition exacerbates native shrub loss and nonnative plant invasion during extreme drought                                                                                                                          | Global Change Biology                                                           | 2017.0            | 10.1111/gcb.13694                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Reconciling seasonal hydraulic risk and plant water use through probabilistic soil–plant dynamics                                                                                                                                                        | Global Change Biology                                                           | 2017.0            | 10.1111/gcb.13640                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Stem photosynthesis and hydraulics are coordinated in desert plant species                                                                                                                                                                               | New Phytologist                                                                 | 2017.0            | 10.1111/nph.14737                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Trade-offs between water transport capacity and drought resistance in neotropical canopy liana and tree species                                                                                                                                          | Tree Physiology                                                                 | 2017.0            | 2-s2.0-85032741600                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Multiple strategies for drought survival among woody plant species                                                                                                                                                                                       | Functional Ecology                                                              | 2016.0            | 10.1111/1365-2435.12518            |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Plant hydraulic responses to long-term dry season nitrogen deposition alter drought tolerance in a Mediterranean-type ecosystem                                                                                                                          | Oecologia                                                                       | 2016.0            | 2-s2.0-84961655125                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Plant hydraulics as a central hub integrating plant and ecosystem function: meeting report for ‘Emerging Frontiers in Plant Hydraulics’ (Washington, DC, May 2015)                                                                                       | Plant Cell and Environment                                                      | 2016.0            | 2-s2.0-84978886186                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Testing the 'microbubble effect' using the Cavitron technique to measure xylem water extraction curves                                                                                                                                                   | AoB PLANTS                                                                      | 2016.0            | 10.1093/aobpla/plw011              |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Using leaf δ<sup>13</sup>C and photosynthetic parameters to understand acclimation to irradiance and leaf age effects during tropical forest regeneration                                                                                                | Forest Ecology and Management                                                   | 2016.0            | 10.1016/j.foreco.2016.07.048       |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Extractable nitrogen and microbial community structure respond to grassland restoration regardless of historical context and soil composition                                                                                                            | AoB PLANTS                                                                      | 2015.0            | 10.1093/aobpla/plu085              |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Global effects of soil and climate on leaf photosynthetic traits and rates                                                                                                                                                                               | Global Ecology and Biogeography                                                 | 2015.0            | 2-s2.0-84929023405                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Global-scale environmental control of plant photosynthetic capacity                                                                                                                                                                                      | Ecological Applications                                                         | 2015.0            | 10.1890/14-2111.1                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Global-scale environmental control of plant photosynthetic capacity                                                                                                                                                                                      | Ecological Applications                                                         | 2015.0            | 10.1890/14-2111.1.sm               |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Lianas always outperform tree seedlings regardless of soil nutrients: Results from a long-term fertilization experiment                                                                                                                                  | Ecology                                                                         | 2015.0            | 2-s2.0-84937022123                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Nutrient limitation of eco-physiological processes in tropical trees                                                                                                                                                                                     | Trees - Structure and Function                                                  | 2015.0            | 10.1007/s00468-015-1260-x          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | PHYSIOLOGICAL IMPLICATIONS OF THE LIANA GROWTH FORM                                                                                                                                                                                                      | Ecology of Lianas                                                               | 2015.0            | WOS:000359791600022                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Rapid recovery of photosynthesis and water relations following soil drying and re-watering is related to the adaptation of desert shrub Ephedra alata subsp. alenda (Ephedraceae) to arid environments                                                   | Environmental and Experimental Botany                                           | 2015.0            | 2-s2.0-84930683111                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Strong phylogenetic signals and phylogenetic niche conservatism in ecophysiological traits across divergent lineages of Magnoliaceae                                                                                                                     | Scientific Reports                                                              | 2015.0            | WOS:000358014900001                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Biogeomorphology of a Mojave Desert landscape - Configurations and feedbacks of abiotic and biotic land surfaces during landform evolution                                                                                                               | Geomorphology                                                                   | 2014.0            | 2-s2.0-84894962097                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Can vessel dimension explain tolerance toward fungal vascular wilt diseases in woody plants? Lessons from dutch elm disease and esca disease in grapevine                                                                                                | Frontiers in Plant Science                                                      | 2014.0            | 10.3389/fpls.2014.00253            |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Contrasting physiological responses of ozone-tolerant Phaseolus vulgaris and Nicotiana tobaccum varieties to ozone and nitric acid                                                                                                                       | Environmental Science: Processes and Impacts                                    | 2014.0            | 2-s2.0-84908161337                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Coordination of stem and leaf hydraulic conductance in southern California shrubs: A test of the hydraulic segmentation hypothesis                                                                                                                       | New Phytologist                                                                 | 2014.0            | WOS:000339556300014                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Determinants of change in subtropical tree diameter growth with ontogenetic stage                                                                                                                                                                        | Oecologia                                                                       | 2014.0            | 10.1007/s00442-014-2981-z          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Light use efficiency of California redwood forest understory plants along a moisture gradient                                                                                                                                                            | Oecologia                                                                       | 2014.0            | WOS:000330734100004                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Physiological implications of the liana growth form                                                                                                                                                                                                      | Ecology of Lianas                                                               | 2014.0            | 2-s2.0-84926456842                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Prometheus wiki gold leaf protocol: Gas exchange using LI-COR 6400                                                                                                                                                                                       | Functional Plant Biology                                                        | 2014.0            | 10.1071/FP10900                    |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Stem, root, and older leaf N: P ratios are more responsive indicators of soil nutrient availability than new foliage                                                                                                                                     | Ecology                                                                         | 2014.0            | 2-s2.0-84905639691                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Stem, root, and older leaf N:P ratios are more responsive indicators of soil nutrient availability than new foliage                                                                                                                                      | Ecology                                                                         | 2014.0            | WOS:000340215900005                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Biological soil crust community types differ in key ecological functions                                                                                                                                                                                 | Soil Biology and Biochemistry                                                   | 2013.0            | WOS:000323686800020                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Ecological role of hybridization in adaptive radiations:A case study in the Dubautia arborea-Dubautia ciliolata (Asteraceae) complex                                                                                                                     | International Journal of Plant Sciences                                         | 2013.0            | WOS:000319056100003                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Exotic annuals reduce soil heterogeneity in coastal sage scrub soil chemical and biological characteristics                                                                                                                                              | Soil Biology and Biochemistry                                                   | 2013.0            | 10.1016/j.soilbio.2012.09.028      |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Forest dynamics of a subtropical monsoon forest in Dinghushan, China: Recruitment, mortality and the pace of community change                                                                                                                            | Journal of Tropical Ecology                                                     | 2013.0            | 10.1017/S0266467413000059          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Source water, phenology and growth of two tropical dry forest tree species growing on shallow karst soils                                                                                                                                                | Trees - Structure and Function                                                  | 2013.0            | 2-s2.0-84884207255                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Nutrients limit photosynthesis in seedlings of a lowland tropical forest tree species                                                                                                                                                                    | Oecologia                                                                       | 2012.0            | WOS:000301705700002                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Tropical tree seedling growth responses to nitrogen, phosphorus and potassium addition                                                                                                                                                                   | Journal of Ecology                                                              | 2012.0            | WOS:000300500800002                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Consequences of light absorptance in calculating electron transport rate of desert and succulent plants                                                                                                                                                  | Photosynthetica                                                                 | 2011.0            | 2-s2.0-79959780466                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Global patterns of leaf mechanical properties                                                                                                                                                                                                            | Ecology Letters                                                                 | 2011.0            | 2-s2.0-79951772076                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Nonparametric tests for homogeneity of species assemblages: A data depth approach                                                                                                                                                                        | Biometrics                                                                      | 2011.0            | WOS:000298095900032                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Oceanographic anomalies and sea-level rise drive mangroves inland in the Pacific coast of Mexico                                                                                                                                                         | Journal of Vegetation Science                                                   | 2011.0            | 2-s2.0-78651359774                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Physiological variation in Hawaiian metrosideros polymorpha across a range of habitats: From dry forests to cloud forests                                                                                                                                | Tropical Montane Cloud Forests: Science for Conservation and Management         | 2011.0            | 2-s2.0-84933564480                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Plant water status and hydraulic conductance during flowering in the southern California coastal sage shrub Salvia mellifera (Lamiaceae)                                                                                                                 | American Journal of Botany                                                      | 2011.0            | WOS:000293513400018                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Potassium, phosphorus, or nitrogen limit root allocation, tree growth, or litter production in a lowland tropical forest                                                                                                                                 | Ecology                                                                         | 2011.0            | 2-s2.0-79961072828                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Potassium, phosphorus, or nitrogen limit root allocation, tree growth, or litter production in a lowland tropical forest                                                                                                                                 | Ecology                                                                         | 2011.0            | WOS:000293459100008                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | A unique web resource for physiology, ecology and the environmental sciences: PrometheusWiki                                                                                                                                                             | Functional Plant Biology                                                        | 2010.0            | WOS:000280294500002                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Belowground nitrogen dynamics in relation to hurricane damage along a tropical dry forest chronosequence                                                                                                                                                 | Biogeochemistry                                                                 | 2010.0            | 10.1007/s10533-009-9378-9          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Can growth form classification predict litter nutrient dynamics and decomposition rates in lowland wet forest?                                                                                                                                           | Biotropica                                                                      | 2010.0            | 2-s2.0-73649108889                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Carbon stable isotopic composition of soluble sugars in Tillandsia epiphytes varies in response to shifts in habitat                                                                                                                                     | Oecologia                                                                       | 2010.0            | 10.1007/s00442-010-1577-5          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Compensatory growth responses to defoliation and light availability in two native Mexican woody plant species                                                                                                                                            | Journal of Tropical Ecology                                                     | 2010.0            | 2-s2.0-77952480118                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Environmental regulation of carbon isotope composition and crassulacean acid metabolism in three plant communities along a water availability gradient                                                                                                   | Oecologia                                                                       | 2010.0            | 10.1007/s00442-010-1724-z          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Hydraulic constraints on photosynthesis in subtropical evergreen broad leaf forest and pine woodland trees of the Florida Everglades                                                                                                                     | Trees - Structure and Function                                                  | 2010.0            | 2-s2.0-77952423043                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | The incidence of crassulacean acid metabolism in Orchidaceae derived from carbon isotope ratios: A checklist of the flora of Panama and Costa Rica                                                                                                       | Botanical Journal of the Linnean Society                                        | 2010.0            | 2-s2.0-77954650976                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Water relations of evergreen and drought-deciduous trees along a seasonally dry tropical forest chronosequence                                                                                                                                           | Oecologia                                                                       | 2010.0            | 10.1007/s00442-010-1725-y          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Correlated evolution of leaf shape and physiology in the woody Sonchus alliance (Asteraceae: Sonchinae) in Macaronesia                                                                                                                                   | International Journal of Plant Sciences                                         | 2009.0            | 2-s2.0-58849161773                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Crassulacean acid metabolism and epiphytism linked to adaptive radiations in the Orchidaceae<sup>1[OA]</sup>                                                                                                                                             | Plant Physiology                                                                | 2009.0            | 2-s2.0-65249115785                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Fog interception by Sequoia sempervirens (D. Don) crowns decouples physiology from soil water deficit                                                                                                                                                    | Plant, Cell and Environment                                                     | 2009.0            | WOS:000266601600011                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Why are non-photosynthetic tissues generally <sup>13</sup>C enriched compared with leaves in C<sub>3</sub> plants? Review and synthesis of current hypotheses                                                                                            | Functional Plant Biology                                                        | 2009.0            | 10.1071/FP08216                    |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Growth, survival and herbivory of seedlings in Brosimum alicastrum (Moraceae), a species from the Neotropical undergrowth                                                                                                                                | Revista De Biologia Tropical                                                    | 2008.0            | WOS:000265268200036                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Growth, survival and herbivory of seedlings in Brosimum alicastrum (Moraceae), a species from the Neotropical undergrowth,Crecimiento, supervivencia y herbivoría de plántulas de Brosimum alicastrum (Moraceae), una especie del sotobosque neotropical | Revista de Biologia Tropical                                                    | 2008.0            | 2-s2.0-70350026853                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Plant species traits are the predominant control on litter decomposition rates within biomes worldwide                                                                                                                                                   | Ecology Letters                                                                 | 2008.0            | 2-s2.0-51249083428                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | A review of volatile analytical methods for determining the botanical origin of honey                                                                                                                                                                    | Food Chemistry                                                                  | 2007.0            | 10.1016/j.foodchem.2006.07.068     |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Extending the leaf economics spectrum to decomposition: Evidence from a tropical forest                                                                                                                                                                  | Ecology                                                                         | 2007.0            | 2-s2.0-34447094487                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Leaf functional traits of tropical forest plants in relation to growth form                                                                                                                                                                              | Functional Ecology                                                              | 2007.0            | WOS:000243412200003                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Nighttime transpiration in woody plants from contrasting ecosystems                                                                                                                                                                                      | Tree Physiology                                                                 | 2007.0            | 2-s2.0-34247392440                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Nighttime transpiration in woody plants from contrasting ecosystems                                                                                                                                                                                      | Tree Physiology                                                                 | 2007.0            | WOS:000246124800010                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | A comparison of sap flow measurements and potometry in two tropical lowland tree species with contrasting wood properties                                                                                                                                | Revista de Biologia Tropical                                                    | 2006.0            | 10.15517/rbt.v54i1.14000           |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | A comparison of sap flow measurements and potometry in two tropical lowland tree species with contrasting wood properties                                                                                                                                | Revista De Biologia Tropical                                                    | 2006.0            | WOS:000243514100009                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Distribution of crassulacean acid metabolism in orchids of Panama: Evidence of selection for weak and strong modes                                                                                                                                       | Functional Plant Biology                                                        | 2005.0            | 10.1071/FP04179                    |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Leaf productivity along a precipitation gradient in lowland Panama: Patterns from leaf to ecosystem                                                                                                                                                      | Trees - Structure and Function                                                  | 2005.0            | WOS:000229510100014                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Nutrient cycling and plant-soil feedbacks along a precipitation gradient in lowland Panama                                                                                                                                                               | Journal of Tropical Ecology                                                     | 2005.0            | 2-s2.0-22744443918                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Use of stable isotopes in tropical biology                                                                                                                                                                                                               | Interciencia                                                                    | 2005.0            | WOS:000231969800004                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Use of stable isotopes in tropical biology,El uso de isótopos estables en biologia tropical                                                                                                                                                              | Interciencia                                                                    | 2005.0            | 2-s2.0-27844579673                 |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Coordinated changes in photosynthesis, water relations and leaf nutritional traits of canopy trees along a precipitation gradient in lowland tropical forest                                                                                             | Oecologia                                                                       | 2004.0            | WOS:000221235800002                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | Leaf photosynthetic traits scale with hydraulic conductivity and wood density in Panamanian forest canopy trees                                                                                                                                          | Oecologia                                                                       | 2004.0            | 10.1007/s00442-004-1624-1          |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID,Scopus - Elsevier            | A test of gas exchange measurements on excised canopy branches of ten tropical tree species                                                                                                                                                              | Photosynthetica                                                                 | 2003.0            | WOS:000188413400004                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Evolution of photosynthetic pathways in the Orchidaceae: Evidence from stable isotopes and phylogenetic analysis.                                                                                                                                        | Ecological Society of America Annual Meeting Abstracts                          | 2003.0            | BCI:BCI200300540854                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Linking plant physiological ecology to ecosystem science: Effects of life history traits on leaf decomposition.                                                                                                                                          | Ecological Society of America Annual Meeting Abstracts                          | 2003.0            | BCI:BCI200300540792                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | ResearcherID                              | Photosynthetic capacity and associated leaf traits along a precipitation gradient in lowland tropical forest                                                                                                                                             | Ecological Society of America Annual Meeting Abstracts                          | 2001.0            | BCI:BCI200200277295                |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Morphological and physiological responses of Hawaiian Hibiscus tiliaceus populations to light and salinity                                                                                                                                               | International Journal of Plant Sciences                                         | 2000.0            | 2-s2.0-0034003068                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Transpiration and forest structure in relation to soil waterlogging in a Hawaiian montane cloud forest                                                                                                                                                   | Tree Physiology                                                                 | 2000.0            | 2-s2.0-0034190632                  |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 | Scopus - Elsevier                         | Use of coarse woody debris by the plant community of a Hawaiian montane cloud forest                                                                                                                                                                     | Biotropica                                                                      | 2000.0            | 10.1111/j.1744-7429.2000.tb00510.x |

**Louis Santiago from the 1992 class funding: (2)**

| input          | given   | family   | orcid               | title   | funder   | identifier   | award   | start   | end   |
|:---------------|:--------|:---------|:--------------------|:--------|:---------|:-------------|:--------|:--------|:------|
| Louis Santiago | Louis   | Santiago | 0000-0002-6943-7177 |         |          |              |         |         |       |
| Louis Santiago | Louis   | Santiago | 0000-0001-5994-6122 |         |          |              |         |         |       |